# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AdityaAAND/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pyspark.sql import SparkSession

# Create a local Spark session using all available cores
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Colab_Spark_Test") \
    .getOrCreate()

# Verify the session
print(spark)



## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from huggingface_hub import HfApi
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
api = HfApi(token=hf_token)



from datasets import load_dataset

ds = load_dataset("FlyRank/internship-warehouse", "fact_content_daily_performance",  split="train")
df = ds.to_pandas()
import duckdb
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
con.sql("SELECT COUNT(*) FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')")

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 19.6kB            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  134MB            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 90.3MB            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 8.93MB            

...

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 3.22MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/78835655 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

In [5]:
# Split design check — time-aware, single global cutoff
import pandas as pd

# 1. Pick the cutoff date: midpoint of the full data's date range
min_date = df["date"].min()
max_date = df["date"].max()
cutoff_date = min_date + (max_date - min_date) / 2

print(f"Data range: {min_date.date()} to {max_date.date()}")
print(f"Cutoff date: {cutoff_date.date()}")

# 2. Split into feature window (before cutoff) and label window (after cutoff)
feature_window = df[df["date"] < cutoff_date]
label_window = df[df["date"] >= cutoff_date]

print(f"Feature window rows: {len(feature_window):,}  ({feature_window['date'].min().date()} to {feature_window['date'].max().date()})")
print(f"Label window rows:   {len(label_window):,}  ({label_window['date'].min().date()} to {label_window['date'].max().date()})")

# 3. Sanity check: no date overlap across the cutoff
overlap = feature_window["date"].max() >= label_window["date"].min()
print(f"Any date overlap across split? {overlap}")  # must print False

# 4. Sanity check: every pair has at least some data on both sides
# (pairs with zero data pre- or post-cutoff can't be used — drop and report count)
pairs_with_both = (
    df.groupby("pair_id")["date"]
    .agg(lambda dates: (dates < cutoff_date).any() and (dates >= cutoff_date).any())
)
print(f"Pairs with data on both sides of cutoff: {pairs_with_both.sum():,} / {len(pairs_with_both):,}")

TypeError: 'generator' object is not subscriptable

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.